# 3.12 · 流水线 / Pipelines

> **课程定位 / Where this fits**
> 第 12 课，**Part 3 · EDA 与数据预处理**（收官）。
> Lesson 12, **Part 3 · EDA & Preprocessing** (finale).
>
> 前 11 课学了一堆预处理步骤（填补、缩放、编码、特征工程、重采样），也反复强调"只能 fit 训练集"。**Pipeline** 把所有这些步骤和模型**打包成一个对象**，从结构上一劳永逸地**防泄漏**，还让调参、保存、部署变得极其干净。这是把 Part 3 所有知识串起来的"总开关"。
> The first 11 lessons taught many preprocessing steps and kept insisting "fit on train only". A **Pipeline** bundles all those steps with the model into **one object**, making leakage **structurally impossible** and turning tuning, saving, and deployment clean. It's the master switch tying all of Part 3 together.
>
> 💼 **实战/面试视角**："为什么用 Pipeline / ColumnTransformer 干嘛的 / 怎么防止预处理泄漏" 是实战素养的体现。
> 💼 **Practical/interview angle:** "why Pipelines / what's ColumnTransformer / how to prevent preprocessing leakage" signal practical maturity.

> 💡 **面试相关 / Interview-relevant**
> - "Pipeline 怎么防止数据泄漏"（出镜率 ★★★★★）
> - "ColumnTransformer 解决什么（数值/类别列分别处理）"（★★★★★）
> - "怎么对 Pipeline 里的步骤调参（双下划线语法）"（★★★★）
> - "怎么保存和部署整条流水线"（★★★★）

---

## 学习目标 / Learning Objectives

1. 用 **Pipeline** 把预处理+模型打包，结构性防泄漏。
   Bundle preprocessing+model in a **Pipeline**, leakage-proof by construction.
2. 用 **ColumnTransformer** 对数值/类别列分别处理。
   Use **ColumnTransformer** to process numeric/categorical columns differently.
3. 用**双下划线语法**对流水线任意步骤一起调参。
   Tune any step of a pipeline with the **double-underscore syntax**.
4. 用 **joblib** 保存整条流水线、对原始数据直接部署。
   Save the whole pipeline with **joblib** and deploy on raw data.
5. 写**自定义转换器**集成特征工程，串起整个 Part 3。
   Write a **custom transformer** for feature engineering, tying all of Part 3 together.

## 目录 / TOC
1. [先建直觉：为什么要 Pipeline ⭐](#1)
2. [最简 Pipeline + 防泄漏 ⭐](#2)
3. [ColumnTransformer：分列处理 ⭐](#3)
4. [对流水线调参（双下划线）⭐](#4)
5. [保存与部署 ⭐](#5)
6. [自定义转换器：串起 Part 3 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉：为什么要 Pipeline ⭐ / Intuition: Why Pipelines

不用 Pipeline 时，你会手动写一长串：填补 → 缩放 → 编码 → 训练，每一步都要**小心翼翼地只 fit 训练集、再 transform 测试集**。问题是：步骤一多就容易**手滑**——某一步不小心在全数据上 fit 了，泄漏就发生了，而且很难发现。
Without a Pipeline you manually chain: impute → scale → encode → train, each step requiring **careful "fit on train, transform on test"**. The problem: with many steps it's easy to **slip** — accidentally fit one step on all data, and leakage happens, hard to spot.

**Pipeline 把这串步骤变成一个对象**。当你对它调用 `.fit(X_train)`，它会按顺序 fit 每一步；调用 `.predict(X_test)` 时，它只 transform 不 fit。在交叉验证里，**每一折都会把整条流水线重新 fit 一遍（只在该折的训练部分）**——于是缩放的 μ/σ、填补的中位数、编码的词表全都只来自训练折，**泄漏在结构上就不可能发生**。
**A Pipeline turns the chain into one object.** Calling `.fit(X_train)` fits each step in order; calling `.predict(X_test)` only transforms. Inside cross-validation, **each fold re-fits the entire pipeline (on that fold's training part only)** — so scaling's μ/σ, imputation's medians, encoding's vocabulary all come from the training fold, making leakage **structurally impossible**.


<a id="2"></a>
## 2. 最简 Pipeline + 防泄漏 ⭐ / Minimal Pipeline & Leakage-proofing

一个三步流水线：填补 → 缩放 → 分类。重点看交叉验证时它如何自动防泄漏。
A three-step pipeline: impute → scale → classify. Note how it auto-prevents leakage during CV.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X = X.copy(); X.iloc[::20, 0] = np.nan        # 人为制造一些缺失 / inject missingness

# 三步流水线: 每步是 (名字, 转换器/模型) / pipeline = list of (name, step)
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # 第1步: 中位数填补
    ("scale", StandardScaler()),                     # 第2步: 标准化
    ("clf", LogisticRegression(max_iter=2000)),      # 最后一步: 分类器
])

# cross_val_score 对每折: 在 train 折 fit 整条流水线, 在 val 折只 transform+predict
scores = cross_val_score(pipe, X, y, cv=5)
print(f"Pipeline CV 准确率: {scores.mean():.1%} ± {scores.std():.1%}")
print("\n每个 fold 内部: ① 在 train 折 fit impute+scale+clf  ② 在 val 折只 transform+predict")
print("→ 填补的中位数、缩放的 μ/σ 都只来自 train 折 → 零泄漏(无需手动操心)")


<a id="3"></a>
## 3. ColumnTransformer：分列处理 ⭐ / ColumnTransformer

真实数据里**数值列和类别列要用不同的处理**：数值列填中位数+缩放，类别列填众数+One-Hot。**ColumnTransformer** 让你给不同的列指定不同的子流水线，再合并成一个统一的预处理器。
Real data needs **different handling for numeric vs categorical columns**: numeric → median impute + scale; categorical → mode impute + One-Hot. **ColumnTransformer** lets you assign different sub-pipelines to different columns and merge them into one preprocessor.


In [ ]:
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df = sns.load_dataset("titanic")
X = df[["pclass","sex","age","sibsp","parch","fare","embarked"]].copy()
y = df["survived"]
num_cols = ["age","sibsp","parch","fare"]        # 数值列 / numeric
cat_cols = ["pclass","sex","embarked"]            # 类别列 / categorical

# 数值子流水线: 中位数填补 → 标准化 / numeric sub-pipeline
num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
# 类别子流水线: 众数填补 → one-hot(未见类别忽略) / categorical sub-pipeline
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])

# ColumnTransformer: 把不同列路由到不同子流水线 / route columns to sub-pipelines
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),     # num_cols 走 num_pipe
    ("cat", cat_pipe, cat_cols),     # cat_cols 走 cat_pipe
])

# 完整流水线: 预处理 + 模型 / preprocessing + model
full = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
print(f"ColumnTransformer + LogReg CV: {cross_val_score(full, X, y, cv=5).mean():.1%}")
print(f"处理后特征数: {preprocess.fit_transform(X).shape[1]} (4 数值 + one-hot 展开的类别列)")


<a id="4"></a>
## 4. 对流水线调参（双下划线）⭐ / Tuning a Pipeline

Pipeline 的强大之处：**整条流水线的任何超参都能一起用 GridSearchCV 调**——包括"填补用均值还是中位数"这种预处理选择，和模型超参一起搜。语法是用**双下划线 `__`** 一层层指定路径。
The power of Pipelines: **any hyperparameter anywhere in the chain can be tuned together with GridSearchCV** — including preprocessing choices like "mean vs median imputation", alongside model hyperparameters. The syntax uses **double-underscore `__`** to drill down the path.


In [ ]:
from sklearn.model_selection import GridSearchCV

full = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
# 双下划线路径: prep(ColumnTransformer)→num(子流水线)→impute(填补器)→strategy(参数)
param_grid = {
    "prep__num__impute__strategy": ["median", "mean"],   # 调数值填补策略(嵌套3层)
    "clf__C": [0.1, 1.0, 10.0],                           # 调模型正则强度
}
grid = GridSearchCV(full, param_grid, cv=5, scoring="accuracy").fit(X, y)
print(f"最优参数 best params: {grid.best_params_}")
print(f"最优 CV 准确率: {grid.best_score_:.1%}")
print("\n语法 prep__num__impute__strategy 含义:")
print("  prep(ColumnTransformer) → num(数值子流水线) → impute(填补器) → strategy(参数)")
print("→ 整个流程任意超参一起调, 每个组合都在 CV 折内诚实评估(无泄漏)")


<a id="5"></a>
## 5. 保存与部署 ⭐ / Saving & Deployment

把整条训练好的流水线用 **joblib** 存成一个文件。部署时只需 `load` + `predict`，**直接喂原始数据**——填补、缩放、编码全封装在里面，永远不会出现"上线时忘了缩放"这种灾难。这是 Pipeline 最大的工程价值。
Save the entire trained pipeline to one file with **joblib**. To deploy, just `load` + `predict`, **feeding raw data** — imputation, scaling, encoding are all encapsulated, so the "forgot to scale in production" disaster can't happen. This is a Pipeline's biggest engineering payoff.


In [ ]:
import joblib
from pathlib import Path

final_model = grid.best_estimator_      # 用最优超参的整条流水线
final_model.fit(X, y)                    # 用全部数据重训(部署前的标准做法)

# 保存整条流水线(预处理逻辑也一并存进去) / save the whole pipeline
joblib.dump(final_model, "/tmp/titanic_pipeline.joblib")
print(f"已保存 saved: /tmp/titanic_pipeline.joblib ({Path('/tmp/titanic_pipeline.joblib').stat().st_size/1024:.0f} KB)")

# 模拟部署: 加载 + 对一条"原始"新数据预测 / deploy: load + predict on RAW new data
loaded = joblib.load("/tmp/titanic_pipeline.joblib")
new_passenger = pd.DataFrame([{"pclass":1,"sex":"female","age":28,"sibsp":0,"parch":0,"fare":80.0,"embarked":"S"}])
prob = loaded.predict_proba(new_passenger)[0, 1]   # 直接喂原始数据, Pipeline 自动处理一切
print(f"\n新乘客(一等舱女性) 生还概率: {prob:.1%}")
print("→ 部署只需 load + predict(原始数据); 预处理封装在内, 永不会'忘了缩放'")


<a id="6"></a>
## 6. 自定义转换器：串起 Part 3 ⭐ / Custom Transformer: All of Part 3

3.6 的特征工程怎么放进 Pipeline？写一个**自定义转换器**——继承 `BaseEstimator` + `TransformerMixin`，实现 `fit`/`transform`。把它放在流水线最前面，就能让"特征工程"也享受 Pipeline 的防泄漏和部署便利。下面这个 `deployable` 对象**串起了整个 Part 3**。
How do you put 3.6's feature engineering into a Pipeline? Write a **custom transformer** — subclass `BaseEstimator` + `TransformerMixin`, implement `fit`/`transform`. Put it at the front, and feature engineering also gets leakage-proofing and deployment. The `deployable` object below **ties all of Part 3 together.**


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# 自定义转换器: 加 3.6 的工程特征 / custom feature-engineering step
class TitanicFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self          # 无需学参数, 直接返回 self
    def transform(self, X):
        X = X.copy()
        X["family_size"] = X["sibsp"] + X["parch"] + 1                    # 3.6: 家庭规模
        X["is_alone"] = (X["family_size"] == 1).astype(int)               # 3.6: 是否独自一人
        X["fare_log"] = np.log1p(X["fare"].fillna(X["fare"].median()))    # 3.6: log 变换右偏
        return X

X = df[["pclass","sex","age","sibsp","parch","fare","embarked"]].copy()
y = df["survived"]
num_cols = ["age","sibsp","parch","fare","family_size","fare_log"]
cat_cols = ["pclass","sex","embarked","is_alone"]

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

# 端到端可部署流水线: 特征工程 → 预处理 → 模型 / fully deployable end-to-end
deployable = Pipeline([
    ("features", TitanicFeatures()),                            # 3.6 特征工程
    ("prep", prep),                                             # 3.2 填补 + 3.4 缩放 + 3.5 编码
    ("clf", RandomForestClassifier(n_estimators=200, random_state=0)),
])
scores = cross_val_score(deployable, X, y, cv=StratifiedKFold(5))   # 3.10 分层CV诚实评估
print(f"端到端流水线 CV 准确率: {scores.mean():.1%} ± {scores.std():.1%}")
deployable.fit(X, y)
print("\n这一个 'deployable' 对象浓缩了整个 Part 3:")
print("  特征工程(3.6) → 数值[中位数填补(3.2)+标准化(3.4)] + 类别[众数填补(3.2)+one-hot(3.5)]")
print("  → 随机森林; 分层CV评估(3.10); 结构防泄漏(3.9); joblib 一行上线")


<a id="7"></a>
## 7. 小结 / Summary

```
Pipeline: 把预处理+模型打包成一个对象; CV 时每折重 fit 整条 → 结构性防泄漏(无需手动操心)
ColumnTransformer: 数值列(填补+缩放) / 类别列(填补+one-hot) 分别处理再合并
调参: GridSearchCV + 双下划线 prep__num__impute__strategy, 预处理选择和模型超参一起搜
部署: joblib.dump 整条流水线; 上线只需 load+predict(原始数据), 预处理封装在内
自定义转换器: 继承 BaseEstimator+TransformerMixin, 把特征工程也纳入流水线
一个 deployable 对象 = 整个 Part 3 (EDA洞察→特征→填补→缩放→编码→模型→评估→上线)
```

### 💡 面试速查 / Interview cheat-sheet
1. **Pipeline 结构性防泄漏**：CV 每折只在 train 部分 fit 所有预处理。
   Pipelines are leakage-proof by construction: each CV fold fits preprocessing on train only.
2. **ColumnTransformer** 给数值/类别列不同处理。
   ColumnTransformer applies different handling to numeric/categorical columns.
3. **双下划线语法**调流水线任意步骤的超参。
   Double-underscore syntax tunes any step's hyperparameters.
4. **joblib 存整条流水线**，部署喂原始数据即可。
   joblib saves the whole pipeline; deploy by feeding raw data.
5. **自定义转换器**把特征工程纳入流水线（继承 BaseEstimator+TransformerMixin）。
   Custom transformers bring feature engineering into the pipeline.

### Part 3 完成 🎉
数据预处理实战全部走通：EDA → 缺失/异常 → 缩放/编码/特征工程 → 文本/图像 → 防泄漏 → CV → 不平衡 → 流水线。下一部分 **Part 4 监督学习：回归**，正式开始建模。
Part 3 complete: EDA → missing/outliers → scaling/encoding/feature-engineering → text/image → leakage → CV → imbalance → pipelines. Next, **Part 4 Regression**, where modeling begins.
